In [ ]:
import torch 
import torchvision


In [ ]:
import os 
from pathlib import Path

data_path = Path('oxford-iiit-pet')

for dirpath, dirnames, filenames in os.walk(data_path) : 
    print(f'# of directories : {len(dirnames)} and {len(filenames)} images in {dirpath}')



In [ ]:
weights = torchvision.models.MobileNet_V3_Small_Weights.DEFAULT

auto_transforms = weights.transforms()

In [ ]:
auto_transforms # Resize value is 256 

In [ ]:
train_data = torchvision.datasets.OxfordIIITPet(
    root = '' , 
    split = 'trainval' ,
    download = True , 
    transform = auto_transforms
)
test_data = torchvision.datasets.OxfordIIITPet(
    root = '' ,
    split = 'test' , 
    download = True , 
    transform = auto_transforms
)

In [ ]:
class_names = train_data.classes

class_names[:5] # It's a long list first five elements is enough for confirm.

In [ ]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    dataset = train_data ,
    batch_size = 64 , 
    shuffle = True , 
    pin_memory = torch.cuda.is_available() 
)
test_dataloader = DataLoader(
    dataset = test_data , 
    batch_size = 64 ,
    shuffle = True ,
    pin_memory = torch.cuda.is_available() 
)

In [ ]:
import matplotlib.pyplot as plt
def plot_images(dataloader : DataLoader , data_num : int , class_names : list) :
    images , labels = next(iter(dataloader))
    
    img = images[data_num].permute(1 , 2 , 0)
    label = labels[data_num]

    plt.figure(figsize = (5 , 5))
    plt.imshow(img)
    plt.title(f'Images is {class_names[label]}')
    plt.show()

plot_images(train_dataloader , data_num = 2 , class_names = class_names)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = torchvision.models.mobilenet_v3_small(weights = weights).to(device)

model

In [ ]:
from torchinfo import summary

summary(model = model , input_size = [32 , 3 , 256 , 256] , col_names = ['output_size' , 'num_params' , 'trainable'])

In [ ]:
# the model has 1000 output but ı just wanna len(class_names) numbers output 
# i am going to modify linear layer output.

In [ ]:
for params in model.features.parameters() : 
    params.requires_grad = False

summary(model = model , input_size = [32 , 3 , 256 , 256] , col_names = ['input_size' , 'num_params' , 'trainable'])

In [ ]:
from torch import nn

class_names = train_data.classes

model.classifier[3] = nn.Linear(in_features = 1024 , out_features = len(class_names) , bias = True).to(device)

summary(model = model , input_size = [32 , 3 , 256 , 256] , col_names = ['input_size' , 'num_params' , 'trainable'])

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params = model.parameters() , lr = 0.001 )

In [ ]:
from torchmetrics.classification import MulticlassAccuracy

def train_step(
        model : nn.Module ,
        dataset : torch.utils.data.DataLoader , 
        loss_fn : nn.Module , 
        optimizer : torch.optim.Optimizer , 
        device 
) : 
    
    model.train()

    train_loss , train_acc = 0 , 0 
    correct_predictions = 0
    total_samples = 0
    num_batches = len(dataset)

    for X , y in dataset :
        X , y = X.to(device) , y.to(device)

        y_pred_logits = model(X)

        loss = loss_fn(y_pred_logits , y)
        train_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        y_pred_labels = torch.softmax(input = y_pred_logits , dim = 1).argmax(dim = 1)
        correct_predictions += (y_pred_labels == y).sum().item()
        total_samples += y.size(0)
    
    train_loss /= num_batches
    train_acc = correct_predictions / total_samples

    return train_loss , train_acc


In [ ]:
def test_step(
        model : nn.Module , 
        dataset : torch.utils.data.DataLoader ,
        loss_fn : nn.Module ,
        device

) :
    model.eval()

    test_loss , test_acc = 0 , 0 
    correct_predictions = 0
    total_samples = 0
    num_batches = len(dataset)
    
    with torch.inference_mode() : 
        for X , y  in dataset : 
            X , y = X.to(device) , y.to(device)

            y_pred_logits = model(X)

            loss = loss_fn(y_pred_logits , y).item()
            test_loss += loss

            y_pred_labels = torch.softmax(input = y_pred_logits , dim = 1 ).argmax(dim = 1)
            correct_predictions += (y_pred_labels == y).sum().item()
            total_samples += y.size(0)
        
        test_loss /= num_batches
        test_acc = correct_predictions / total_samples

    return test_loss , test_acc


In [ ]:
def train_test_model(
        model : nn.Module , 
        train_data : torch.utils.data.DataLoader , 
        test_data : torch.utils.data.DataLoader ,
        loss_fn : nn.Module ,
        optimizer : torch.optim.Optimizer ,
        device ,
        epochs : int = 10 
) : 
    for epoch in range(epochs) : 
        train_loss , train_acc = train_step(
            model = model , 
            dataset = train_data , 
            loss_fn = loss_fn , 
            optimizer = optimizer , 
            device = device 
        )
        test_loss , test_acc = test_step(
            model = model , 
            dataset = test_data , 
            loss_fn = loss_fn , 
            device = device
        )

        print(
                f"Epoch: {epoch+1} | "
                f"train_loss: {train_loss:.4f} | "
                f"train_acc: {train_acc:.4f} | "
                f"test_loss: {test_loss:.4f} | "
                f"test_acc: {test_acc:.4f}"
        )

In [ ]:
train_test_model(
    model = model , 
    train_data = train_dataloader , 
    test_data = test_dataloader , 
    loss_fn = loss_fn , 
    optimizer = optimizer , 
    device = device
)

In [ ]:
# Model is overfitted i will modify transforms

In [ ]:
from torchvision import transforms


train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_transforms = transforms.Compose([
    transforms.Resize(256) ,
    transforms.CenterCrop(224),
    transforms.ToTensor() ,
    transforms.Normalize(
        mean = [0.485, 0.456, 0.406] ,
        std = [0.229, 0.224, 0.225]
    )
])

train_data = torchvision.datasets.OxfordIIITPet(
    root = '' , 
    split = 'trainval' ,
    download = True , 
    transform = train_transforms
)
test_data = torchvision.datasets.OxfordIIITPet(
    root = '' ,
    split = 'test' , 
    download = True , 
    transform = test_transforms
)

train_dataloader = DataLoader(
    dataset = train_data , 
    batch_size = 64 ,  
    shuffle = True , 
    pin_memory = torch.cuda.is_available()
)
test_dataloader = DataLoader(
    dataset = test_data , 
    batch_size = 64 ,  
    shuffle = False , 
    pin_memory = torch.cuda.is_available()
)


In [ ]:
weights = torchvision.models.MobileNet_V3_Small_Weights.DEFAULT

model_2 = torchvision.models.mobilenet_v3_small(weights = weights).to(device)

model_2

In [ ]:
summary(model = model_2 , input_size = [32 , 3 , 256 , 256] , col_names = ['input_size' , 'num_params' , 'trainable'])

In [ ]:
for params in model_2.parameters():
    params.requires_grad = False 

model_2.classifier[3] = nn.Linear(in_features=1024, out_features=len(class_names), bias=True).to(device)

#nn.Linear(1024, len(class_names) , bias = True ).to(device)

summary(model = model_2 , input_size = [32 , 3 , 256 , 256] , col_names = ['input_size' , 'num_params' , 'trainable'])


In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model_2.parameters() , lr = 0.001)

train_test_model(
    model = model_2 , 
    train_data = train_dataloader , 
    test_data = test_dataloader , 
    loss_fn = loss_fn , 
    optimizer = optimizer , 
    device = device , 
    epochs = 10 
)
